In [1]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=100, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,
                                                    random_state=1)
clf = MLPClassifier(hidden_layer_sizes=(100,80)).fit(X_train, y_train)

In [2]:
total_params = (
    sum(w.size for w in clf.coefs_) +
    sum(b.size for b in clf.intercepts_)
)

print(total_params)

10261


In [3]:
import torch
def f(x, y):
    return x ** 2 * y + x + y**2

x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
result = f(x, y)
result.backward()

x.grad.item(), y.grad.item()

(7.0, 7.0)

In [4]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

n = 8
d = 5

X = torch.randn(n, d)
y = torch.randint(low=0, high=2, size=(n,)).float()  # 0/1 labels

w = torch.randn(d, requires_grad=True)

logits = X @ w                      # shape (n,)
loss = F.binary_cross_entropy_with_logits(logits, y)

print("logits.shape:", logits.shape)
print("loss:", loss.item())

logits.shape: torch.Size([8])
loss: 1.33346688747406


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

n = 8
d = 5
h = 16

X = torch.randn(n, d)
y = torch.randint(low=0, high=2, size=(n,)).float()

model = nn.Sequential(
    nn.Linear(d, h),
    nn.ReLU(),
    nn.Linear(h, 1),
)

logits = model(X).squeeze(1)     # shape (n,)
loss = F.binary_cross_entropy_with_logits(logits, y)

print("logits.shape:", logits.shape)
print("loss:", loss.item())

logits.shape: torch.Size([8])
loss: 0.6901422142982483


In [6]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

113

In [7]:
import torch

torch.manual_seed(0)


w = torch.tensor(1.0, requires_grad=True)

# define loss：L = (w - 5)^2
loss = (w - 5) ** 2
loss.backward()

print("=== Before update ===")
print("w =", w)
print("w.requires_grad =", w.requires_grad)
print("w.is_leaf =", w.is_leaf)
print("w.grad =", w.grad)

lr = 0.1

# WRONG!!!
w = w - lr * w.grad

print("\n=== After WRONG update ===")
print("w =", w)
print("w.requires_grad =", w.requires_grad)
print("w.is_leaf =", w.is_leaf)
print("w.grad =", w.grad)

# once again forward/backward
loss2 = (w - 5) ** 2
loss2.backward()

print("\n=== After second backward ===")
print("w =", w)
print("w.grad =", w.grad)

=== Before update ===
w = tensor(1., requires_grad=True)
w.requires_grad = True
w.is_leaf = True
w.grad = tensor(-8.)

=== After WRONG update ===
w = tensor(1.8000, grad_fn=<SubBackward0>)
w.requires_grad = True
w.is_leaf = False
w.grad = None

=== After second backward ===
w = tensor(1.8000, grad_fn=<SubBackward0>)
w.grad = None


/tmp/ipykernel_4110/1531898386.py:27: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  print("w.grad =", w.grad)
/tmp/ipykernel_4110/1531898386.py:35: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch

In [8]:
import torch

torch.manual_seed(0)

w = torch.tensor(1.0, requires_grad=True)
lr = 0.1

for step in range(3):
    # clear the gradient to avoid accumulation
    if w.grad is not None:
        w.grad.zero_()

    loss = (w - 5) ** 2
    loss.backward()

    print(f"\n=== Step {step} ===")
    print("before update:")
    print("w =", w.item())
    print("loss =", loss.item())
    print("grad =", w.grad.item())
    print("is_leaf =", w.is_leaf)

    # correct way
    with torch.no_grad():
        w -= lr * w.grad

    print("after update:")
    print("w =", w.item())
    print("is_leaf =", w.is_leaf)


=== Step 0 ===
before update:
w = 1.0
loss = 16.0
grad = -8.0
is_leaf = True
after update:
w = 1.7999999523162842
is_leaf = True

=== Step 1 ===
before update:
w = 1.7999999523162842
loss = 10.24000072479248
grad = -6.400000095367432
is_leaf = True
after update:
w = 2.440000057220459
is_leaf = True

=== Step 2 ===
before update:
w = 2.440000057220459
loss = 6.553599834442139
grad = -5.119999885559082
is_leaf = True
after update:
w = 2.952000141143799
is_leaf = True


In [9]:
import torch

w = torch.tensor(2.0, requires_grad=True)

(0.5 * w**2).backward()
print("after first backward, w.grad =", w.grad.item())   # 2.0

(0.5 * (w - 1.0)**2).backward()
print("after second backward, w.grad =", w.grad.item())  # accumulated

w.grad = None
(0.5 * (w - 1.0)**2).backward()
print("after clearing, w.grad =", w.grad.item())         # correct for the last loss

after first backward, w.grad = 2.0
after second backward, w.grad = 3.0
after clearing, w.grad = 1.0


**Pitfalls and A few PyTorch conventions:**

1.   mark variables with requires_grad=True
2.   clear w.grad each iteration
3.   update parameters under torch.no_grad()